<a href="https://colab.research.google.com/github/alfnihilus/computational-chem/blob/main/creat_molecul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# First, install the PubChemPy and RDKit Python libraries
!pip install pubchempy rdkit

In [ ]:
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem

def generate_molecule(identifier):
    try:
        # 1. Coba cari berdasarkan Nama (e.g., "Water", "Caffeine")
        print(f"Mencari sebagai Nama: {identifier}...")
        results = pcp.get_compounds(identifier, 'name')

        # 2. Jika tidak ketemu, coba cari berdasarkan Rumus (e.g., "H2O", "C2H6O")
        if not results:
            print(f"Tidak ditemukan sebagai nama. Mencari sebagai Rumus: {identifier}...")
            results = pcp.get_compounds(identifier, 'formula')

        if not results:
            print("Molekul tidak ditemukan di database PubChem.")
            return

        # Mengambil hasil pertama yang ditemukan
        comp = results[0]
        # Mengubah dari 'isomeric_smiles' ke 'smiles' untuk menghindari peringatan deprecation
        smiles = comp.smiles
        print(f"Ditemukan! SMILES: {smiles}")

        # 3. Proses RDKit untuk Koordinat 3D
        mol = Chem.MolFromSmiles(smiles)
        mol = Chem.AddHs(mol)

        # Memberikan koordinat 3D awal
        AllChem.EmbedMolecule(mol, AllChem.ETKDG())
        # Optimasi struktur agar stabil (Universal Force Field)
        AllChem.UFFOptimizeMolecule(mol)

        # 4. Cetak koordinat
        print(f"\nKoordinat untuk {identifier}:")
        conf = mol.GetConformer()
        for i, atom in enumerate(mol.GetAtoms()):
            pos = conf.GetAtomPosition(i)
            print(f"{atom.GetSymbol()} {pos.x:.4f} {pos.y:.4f} {pos.z:.4f}")
        print("\nAnda dapat menyalin koordinat di atas untuk penggunaan lebih lanjut.")

    except Exception as e:
        print(f"Error: {e}")

# Input dari user
# @markdown # Pembuatan Koordinat Secara Otomatis (Sumber PubChem)
# @markdown ### Masukkan Nama atau Rumus Kimia (misal: H2O, Ethanol, C6H6, Flavonoid)
user_input ="" # @param {type:"string"}
generate_molecule(user_input)

In [ ]:
# First, install the Ase Python libraries
!pip install ase

In [ ]:
import re
import numpy as np
from ase import Atoms
from ase.io import write

def parse_formula(formula):
    # Memisahkan rumus (misal U79O90C10 menjadi [('U', 79), ('O', 90), ('C', 10)])
    elements = re.findall(r'([A-Z][a-z]*)(\d*)', formula)
    parsed = []
    for el, count in elements:
        count = int(count) if count else 1
        parsed.extend([el] * count)
    return parsed

def generate_random_cluster(formula, density=0.5):
    atom_list = parse_formula(formula)
    num_atoms = len(atom_list)

    # Menghitung perkiraan ukuran kotak berdasarkan jumlah atom
    box_size = (num_atoms / density) ** (1/3)

    # Membuat koordinat acak (x, y, z)
    positions = np.random.rand(num_atoms, 3) * box_size

    # Membuat objek atom ASE
    cluster = Atoms(symbols=atom_list, positions=positions)

    # Optimasi jarak antar atom sederhana agar tidak bertumpukan (Push-apart)
    # Catatan: Ini bukan optimasi kimia, hanya agar tidak overlap di GaussView
    cluster.center(vacuum=5.0)

    # Cetak koordinat
    print(f"\nKoordinat acak untuk {formula} dengan {num_atoms} atom:")
    for i, symbol in enumerate(cluster.get_chemical_symbols()):
        pos = cluster.get_positions()[i]
        print(f"{symbol} {pos[0]:.4f} {pos[1]:.4f} {pos[2]:.4f}")
    print("\nAnda dapat menyalin koordinat di atas untuk penggunaan lebih lanjut.")

# Contoh penggunaan
# @markdown # Pembuatan Molekul Manual
# @markdown ### Masukkan rumus kimia acak (misal U10O20C5)
rumus ="" # @param {type:"string"}
generate_random_cluster(rumus)